In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# If you're in Colab, run this once, then Runtime → Restart runtime.
%pip -q install "tensorflow==2.17.0" "pandas==2.2.2" "scikit-learn==1.5.2" \
                 "opencv-python==4.10.0.84" "tqdm==4.66.4"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 91.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.17.0 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [3]:
!pip install -U numpy==1.26.4
!pip install -U pandas tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.66.4
    Uninstalling tqdm-4.66.4:
      Successfully uninstalled tqdm-4.66.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
dask-cudf-cu12 25

In [6]:
import os, math, json, random
from pathlib import Path
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import xception as xcep

from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_recall_fscore_support, confusion_matrix
)
from sklearn.model_selection import train_test_split

import gradio as gr

# ======== EDIT ME ========
IMG_DIR      = "/content/drive/MyDrive/DeepLearning/faceforensics_benchmark_images"         # folder with 0001.png, 0002.png, ...
WEIGHTS_PATH = "/content/drive/MyDrive/DeepLearning/ffpp_autolabel/xception_ffpp_weights.h5" # your Keras .h5 (binary head)
OUT_DIR      = "/content/drive/MyDrive/DeepLearning/ffpp_autolabel_out"
TARGET_UNCERTAIN = 300     # aim to manually label ~this many (auto-tunes the band width)
# =========================

IMG_EXTS   = {".png",".jpg",".jpeg",".PNG",".JPG",".JPEG"}
IMG_SIZE   = (299, 299)
BATCH_SIZE = 64
SEED = 1234

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("TF:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


TF: 2.17.0
TF: 2.17.0
NumPy: 1.26.4
Pandas: 2.3.3


In [7]:
def list_images(root):
    root = Path(root)
    assert root.exists(), f"IMG_DIR not found: {root}"
    imgs = [p for p in root.rglob("*") if p.is_file() and p.suffix in IMG_EXTS]
    if not imgs:
        raise FileNotFoundError(f"No images found under {root}")
    return sorted(imgs)

imgs = list_images(IMG_DIR)
df = pd.DataFrame({"image_path": [str(p) for p in imgs]})
df["video_id"] = df["image_path"].apply(lambda s: Path(s).stem)
print(f"Found {len(df)} images. Example:\n", df.head())


Found 1000 images. Example:
                                           image_path video_id
0  /content/drive/MyDrive/DeepLearning/faceforens...     0000
1  /content/drive/MyDrive/DeepLearning/faceforens...     0001
2  /content/drive/MyDrive/DeepLearning/faceforens...     0002
3  /content/drive/MyDrive/DeepLearning/faceforens...     0003
4  /content/drive/MyDrive/DeepLearning/faceforens...     0004


In [8]:
def preprocess_img(fp):
    img = tf.io.read_file(fp)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')
    img = xcep.preprocess_input(img)  # [-1,1]
    return img

def make_ds(frame_df, batch_size=BATCH_SIZE):
    paths  = frame_df["image_path"].values
    labels = np.zeros(len(paths), dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p,l: (preprocess_img(p), l), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def build_xception_binary():
    base = xcep.Xception(include_top=False, weights=None, input_shape=(299,299,3))
    x = layers.GlobalAveragePooling2D()(base.output)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(base.input, out)

# load model/weights
try:
    model = tf.keras.models.load_model(WEIGHTS_PATH, compile=False)
    print("Loaded full model:", WEIGHTS_PATH)
except Exception as e:
    print("Falling back to architecture + load_weights:", e)
    model = build_xception_binary()
    model.load_weights(WEIGHTS_PATH)
    print("Loaded weights:", WEIGHTS_PATH)

# inference
ds = make_ds(df)
probs = []
for batch_imgs, _ in tqdm(ds, total=math.ceil(len(df)/BATCH_SIZE), desc="Infer"):
    pr = model.predict(batch_imgs, verbose=0).ravel()
    probs.append(pr)
probs = np.concatenate(probs, axis=0).astype(np.float32)

scores = df.copy()
scores["prob_fake"] = probs
scores.to_csv(f"{OUT_DIR}/all_scores.csv", index=False)
print("Scores saved:", f"{OUT_DIR}/all_scores.csv", "range:", float(probs.min()), "→", float(probs.max()))


Loaded full model: /content/drive/MyDrive/DeepLearning/ffpp_autolabel/xception_ffpp_weights.h5


Infer:   0%|          | 0/16 [00:00<?, ?it/s]

Scores saved: /content/drive/MyDrive/DeepLearning/ffpp_autolabel_out/all_scores.csv range: 0.4999977648258209 → 0.5000030994415283


In [9]:
def gaussian_intersection(m0, s0, m1, s1):
    a = 1/(2*s0*s0) - 1/(2*s1*s1)
    b = m1/(s1*s1) - m0/(s0*s0)
    c = (m0*m0)/(2*s0*s0) - (m1*m1)/(2*s1*s1) - np.log(s1/s0)
    if abs(a)<1e-12:
        return float(-c/b) if abs(b)>1e-12 else float((m0+m1)/2)
    disc = b*b - 4*a*c
    if disc < 0:
        return float((m0+m1)/2.0)
    x1 = (-b + np.sqrt(disc)) / (2*a)
    x2 = (-b - np.sqrt(disc)) / (2*a)
    cand = x1 if (min(m0,m1) <= x1 <= max(m0,m1)) else x2
    return float(np.clip(cand, 0.0, 1.0))

def fit_threshold(scores):
    x = scores.reshape(-1,1)
    try:
        gmm = GaussianMixture(n_components=2, random_state=SEED).fit(x)
        means = gmm.means_.ravel(); stds = np.sqrt(gmm.covariances_.ravel())
        order = np.argsort(means)
        m0, m1 = means[order[0]], means[order[1]]
        s0, s1 = stds[order[0]], stds[order[1]]
        th = gaussian_intersection(m0, s0, m1, s1)
        return th, (m0, s0, m1, s1), "gmm"
    except Exception:
        km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(x)
        c0 = scores[km.labels_==0].mean(); c1 = scores[km.labels_==1].mean()
        th = float((c0 + c1) / 2.0)
        return th, (min(c0,c1), 0.1, max(c0,c1), 0.1), "kmeans"

probs_np = scores["prob_fake"].values.astype(np.float32)
th, (m0,s0,m1,s1), how = fit_threshold(probs_np)
print({"threshold": th, "method": how, "means": [float(m0), float(m1)]})

# choose a band around threshold to get ~TARGET_UNCERTAIN images
# find delta so that count(|score - th| <= delta) ~ target
sorted_abs = np.sort(np.abs(probs_np - th))
if len(sorted_abs) == 0:
    delta = 0.05
else:
    k = np.clip(TARGET_UNCERTAIN, 1, len(sorted_abs))
    delta = float(sorted_abs[int(k)-1])
delta = max(delta, 0.02)  # minimum 0.02 band
print("Uncertainty band width (±delta):", delta)

is_uncertain = np.abs(probs_np - th) <= delta
n_unc = int(is_uncertain.sum())
print("Uncertain images:", n_unc, "of", len(scores))

# auto-label confident tails
auto = scores.copy()
auto["label"] = -1
auto.loc[probs_np >= th + delta, "label"] = 1  # confident fake
auto.loc[probs_np <= th - delta, "label"] = 0  # confident real

auto_labeled = auto[auto["label"]!=-1].copy().reset_index(drop=True)
uncertain   = auto[auto["label"]==-1].copy().reset_index(drop=True)

auto_labeled.to_csv(f"{OUT_DIR}/auto_labeled.csv", index=False)
uncertain.to_csv(f"{OUT_DIR}/uncertain_tolabel.csv", index=False)

print("Auto-labeled:", len(auto_labeled), " → real(0) =", int((auto_labeled.label==0).sum()),
      " fake(1) =", int((auto_labeled.label==1).sum()))
print("To label manually:", len(uncertain))


{'threshold': 0.4583333333333333, 'method': 'gmm', 'means': [0.5000002512931824, 0.5000002512931827]}
Uncertainty band width (±delta): 0.041666507720947266
Uncertain images: 303 of 1000
Auto-labeled: 705  → real(0) = 0  fake(1) = 705
To label manually: 295


In [10]:
OUT_CSV = f"{OUT_DIR}/labels_images.csv"  # final merged labels will be written here (do not run)

# resume: start from existing CSV if present
existing = {}
if Path(OUT_CSV).exists():
    df_old = pd.read_csv(OUT_CSV)
    if {"image_path","label"}.issubset(df_old.columns):
        existing = {row.image_path: int(row.label) for _,row in df_old.iterrows()}

to_label_paths = [p for p in uncertain["image_path"].tolist() if p not in existing]

state = {
    "ordered": to_label_paths,
    "i": 0,
    "labels": existing,  # image_path -> 0/1, prefilled with auto-labeled below after UI ends
    "history": []
}

def status_line():
    done = len(state["labels"])
    total = len(auto_labeled) + len(to_label_paths)
    return f"Labeled (including auto): {done}/{total} ({done/total*100:.1f}%)"

def load_image(idx):
    if len(state["ordered"]) == 0:
        return None, f"Nothing to label. {status_line()}", "All uncertain images labeled or none existed."
    idx = max(0, min(idx, len(state["ordered"]) - 1))
    state["i"] = idx
    img_path = state["ordered"][idx]
    meta = f"{status_line()} | {Path(img_path).name} ({idx+1}/{len(state['ordered'])})"
    hint = "Mark: REAL (0) or FAKE (1). Skip to move on."
    return img_path, meta, hint

def _assign(label):
    if not state["ordered"]:
        return None, status_line(), "No images queued."
    img_path = state["ordered"][state["i"]]
    prev = state["labels"].get(img_path, None)
    state["labels"][img_path] = label
    state["history"].append((img_path, prev))
    return next_image()

def mark_real(): return _assign(0)
def mark_fake(): return _assign(1)

def skip_image():
    return next_image()

def next_image():
    if not state["ordered"]:
        return None, status_line(), "Done."
    i = min(state["i"] + 1, len(state["ordered"]) - 1)
    return load_image(i)

def prev_image():
    if not state["ordered"]:
        return None, status_line(), "Done."
    i = max(state["i"] - 1, 0)
    return load_image(i)

def undo():
    if not state["history"]:
        return load_image(state["i"])
    img_path, prev = state["history"].pop()
    if prev is None:
        state["labels"].pop(img_path, None)
    else:
        state["labels"][img_path] = prev
    try:
        idx = state["ordered"].index(img_path)
    except ValueError:
        idx = state["i"]
    return load_image(idx)

def save_csv():
    # merge: auto-labeled + manually labeled (uncertain)
    auto_rows = auto_labeled[["image_path","label"]].to_dict("records")
    manual_rows = [{"image_path": k, "label": v} for k,v in state["labels"].items()]
    merged = {}
    for r in auto_rows: merged[r["image_path"]] = r["label"]
    for r in manual_rows: merged[r["image_path"]] = r["label"]
    df_out = pd.DataFrame([{"image_path": k, "label": v} for k,v in merged.items()]).sort_values("image_path")
    df_out.to_csv(OUT_CSV, index=False)
    return f"Saved {len(df_out)} total labels to {OUT_CSV}. You can close the UI."

with gr.Blocks(title="Label Uncertain Images (0=Real, 1=Fake)") as demo:
    gr.Markdown("## Label Uncertain Images — 0=Real, 1=Fake")
    with gr.Row():
        img = gr.Image(label="Image", interactive=False)
        with gr.Column():
            meta = gr.Markdown()
            hint = gr.Markdown()
            btn_prev = gr.Button("⟵ Prev")
            btn_next = gr.Button("Next ⟶")
            btn_undo = gr.Button("Undo")
            btn_real = gr.Button("Mark REAL (0)", variant="primary")
            btn_fake = gr.Button("Mark FAKE (1)", variant="primary")
            btn_skip = gr.Button("Skip")
            btn_save = gr.Button("Save CSV")
            saved_msg = gr.Markdown()

    btn_real.click(mark_real, outputs=[img, meta, hint])
    btn_fake.click(mark_fake, outputs=[img, meta, hint])
    btn_skip.click(skip_image, outputs=[img, meta, hint])
    btn_prev.click(prev_image, outputs=[img, meta, hint])
    btn_next.click(next_image, outputs=[img, meta, hint])
    btn_undo.click(undo, outputs=[img, meta, hint])
    btn_save.click(save_csv, outputs=[saved_msg])

    demo.load(lambda: load_image(0), outputs=[img, meta, hint])

print("Launching UI. Label only the uncertain images, then click 'Save CSV'.")
demo.launch(share=False)


Launching UI. Label only the uncertain images, then click 'Save CSV'.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [12]:
final_csv = Path(f"{OUT_DIR}/labels_images.csv")  #(do not run)
if final_csv.exists():
    df_lab = pd.read_csv(final_csv)
    print("Final labels:", final_csv, "rows:", len(df_lab))
    print(df_lab["label"].value_counts().to_dict())
    # You can now set LABELS_CSV to this file in your baseline notebook:
    # LABELS_CSV = f"{OUT_DIR}/labels_images.csv"
    # and run the fully supervised metrics + F1-optimal threshold.
else:
    print("labels_images.csv not found yet — click 'Save CSV' in the UI.")


Final labels: /content/drive/MyDrive/DeepLearning/ffpp_autolabel_out/labels_images.csv rows: 999
{1: 854, 0: 145}


In [13]:
labels_path = Path(OUT_CSV)
if not labels_path.exists():
    raise SystemExit("labels_images.csv not found yet — please click 'Save CSV' in the UI, then re-run this cell.")

df_labels = pd.read_csv(labels_path)
print("Final labels:", labels_path, "rows:", len(df_labels))
print(df_labels["label"].value_counts().to_dict())
assert {"image_path","label"}.issubset(df_labels.columns), "labels_images.csv missing required columns."
assert set(df_labels["label"].unique())=={0,1}, "Both classes required."

# Merge labels with all images; keep only labeled rows for supervised metrics
df_sup = df.merge(df_labels, on="image_path", how="inner")
print("Supervised labeled images:", len(df_sup))

# Stratified 80/10/10 split
train_df, tmp_df = train_test_split(df_sup, test_size=0.20, random_state=SEED, stratify=df_sup["label"])
val_df,   test_df = train_test_split(tmp_df,  test_size=0.50, random_state=SEED, stratify=tmp_df["label"])
for name, part in [("train",train_df),("val",val_df),("test",test_df)]:
    print(name, part["label"].value_counts().to_dict())

# tf.data sets
def make_labeled_ds(frame_df, batch_size=BATCH_SIZE, shuffle=False):
    paths  = frame_df["image_path"].values
    labels = frame_df["label"].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(20000, len(paths)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p,l: (preprocess_img(p), l), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

val_ds  = make_labeled_ds(val_df,  shuffle=False)
test_ds = make_labeled_ds(test_df, shuffle=False)

# Inference helpers
def infer_probs(df_split, ds, model):
    probs = []
    for batch_imgs, _ in tqdm(ds, total=math.ceil(len(df_split)/BATCH_SIZE), desc=f"Infer {len(df_split)}"):
        pr = model.predict(batch_imgs, verbose=0).ravel()
        probs.append(pr)
    probs = np.concatenate(probs, axis=0)
    out = df_split.copy()
    out["prob_fake"] = probs
    return out

def best_threshold_by_f1(y_true, y_score):
    ths = np.linspace(0, 1, 1001)
    best_f1, best_th = -1.0, 0.5
    for th in ths:
        y_pred = (y_score >= th).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1: best_f1, best_th = f1, th
    return float(best_th), float(best_f1)

# VAL → choose threshold
val_out = infer_probs(val_df, val_ds, model)
yv = val_out["label"].values.astype(int)
sv = val_out["prob_fake"].values.astype(np.float32)
th_opt, f1_val = best_threshold_by_f1(yv, sv)
pv = (sv >= th_opt).astype(int)
acc = accuracy_score(yv, pv)
f1  = f1_score(yv, pv)
au  = roc_auc_score(yv, sv)
prec, rec, _, _ = precision_recall_fscore_support(yv, pv, average='binary', zero_division=0)
cm = confusion_matrix(yv, pv, labels=[0,1])

metrics_val = {
    "threshold": th_opt,
    "accuracy": float(acc),
    "f1": float(f1),
    "auroc": float(au),
    "precision": float(prec),
    "recall": float(rec),
    "confusion_matrix": cm.tolist()
}
with open(f"{OUT_DIR}/val_metrics.json", "w") as f: json.dump(metrics_val, f, indent=2)
print("VAL metrics:", json.dumps(metrics_val, indent=2))

# TEST with chosen threshold
test_out = infer_probs(test_df, test_ds, model)
yt = test_out["label"].values.astype(int)
st = test_out["prob_fake"].values.astype(np.float32)
pt = (st >= th_opt).astype(int)

acc = accuracy_score(yt, pt)
f1  = f1_score(yt, pt)
au  = roc_auc_score(yt, st)
prec, rec, _, _ = precision_recall_fscore_support(yt, pt, average='binary', zero_division=0)
cm = confusion_matrix(yt, pt, labels=[0,1])

metrics_test = {
    "threshold": th_opt,
    "accuracy": float(acc),
    "f1": float(f1),
    "auroc": float(au),
    "precision": float(prec),
    "recall": float(rec),
    "confusion_matrix": cm.tolist()
}
with open(f"{OUT_DIR}/test_metrics.json", "w") as f: json.dump(metrics_test, f, indent=2)
print("TEST metrics:", json.dumps(metrics_test, indent=2))

Final labels: /content/drive/MyDrive/DeepLearning/ffpp_autolabel_out/labels_images.csv rows: 999
{1: 854, 0: 145}
Supervised labeled images: 999
train {1: 683, 0: 116}
val {1: 86, 0: 14}
test {1: 85, 0: 15}


Infer 100:   0%|          | 0/2 [00:00<?, ?it/s]

VAL metrics: {
  "threshold": 0.0,
  "accuracy": 0.86,
  "f1": 0.9247311827956989,
  "auroc": 0.8924418604651162,
  "precision": 0.86,
  "recall": 1.0,
  "confusion_matrix": [
    [
      0,
      14
    ],
    [
      0,
      86
    ]
  ]
}


Infer 100:   0%|          | 0/2 [00:00<?, ?it/s]

TEST metrics: {
  "threshold": 0.0,
  "accuracy": 0.85,
  "f1": 0.918918918918919,
  "auroc": 0.9125490196078432,
  "precision": 0.85,
  "recall": 1.0,
  "confusion_matrix": [
    [
      0,
      15
    ],
    [
      0,
      85
    ]
  ]
}


In [14]:
# CSVs (image-level == video-level here)
frame_csv = f"{OUT_DIR}/frame_level.csv"
video_csv = f"{OUT_DIR}/video_level.csv"
(test_out.assign(id=test_out.image_path, prob_fake=test_out.prob_fake,
                 pred_label=(test_out.prob_fake>=th_opt).astype(int))
         [["id","label","prob_fake","pred_label"]]
         .to_csv(frame_csv, index=False))
(test_out.assign(id=test_out.image_path, prob_fake=test_out.prob_fake,
                 pred_label=(test_out.prob_fake>=th_opt).astype(int))
         [["id","label","prob_fake","pred_label"]]
         .to_csv(video_csv, index=False))
print("Wrote CSVs:", frame_csv, "|", video_csv)

# Grad-CAM on validation: top-10 confident fakes and reals
from tensorflow.keras import layers as KL

def find_last_conv_layer(m):
    for layer in reversed(m.layers):
        if isinstance(layer, KL.Conv2D): return layer.name
    raise ValueError("No Conv2D for Grad-CAM.")

def gradcam_heatmap(img_tensor, mdl, last_conv_layer_name, pred_index=0):
    grad_model = tf.keras.models.Model([mdl.inputs],
                                       [mdl.get_layer(last_conv_layer_name).output, mdl.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_tensor)
        loss = preds[:, pred_index]
    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    conv_out = conv_out[0]
    heat = tf.reduce_sum(conv_out * pooled, axis=-1)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()

def overlay_heatmap_on_image(orig_bgr, heatmap, alpha=0.35):
    hm = cv2.resize(heatmap, (orig_bgr.shape[1], orig_bgr.shape[0]))
    hm = np.uint8(255*hm)
    color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    return cv2.addWeighted(color, alpha, orig_bgr, 1-alpha, 0)

OUT_GC = Path(OUT_DIR)/"gradcam_samples"; OUT_GC.mkdir(parents=True, exist_ok=True)
try:
    last_conv = find_last_conv_layer(model)
    vvid = val_out.sort_values("prob_fake", ascending=False)
    top_fakes = vvid.head(10)
    top_reals = val_out.sort_values("prob_fake", ascending=True).head(10)

    def write_gc(row, tag):
        fpath = row.image_path
        bgr = cv2.imread(fpath);
        if bgr is None: return
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, IMG_SIZE)
        x = xcep.preprocess_input(rgb.astype(np.float32))[None,...]
        heat = gradcam_heatmap(tf.convert_to_tensor(x), model, last_conv, pred_index=0)
        over = overlay_heatmap_on_image(bgr, heat)
        cv2.imwrite(str(OUT_GC/f"{tag}_{Path(fpath).stem}_{float(row.prob_fake):.3f}.jpg"), over)

    for r in top_fakes.itertuples(): write_gc(r, "fake")
    for r in top_reals.itertuples(): write_gc(r, "real")
    print("Saved Grad-CAM samples to:", str(OUT_GC))
except Exception as e:
    print("Grad-CAM skipped:", e)

# README
readme = f"""
# FF++ Baseline: Auto-Label Assist → Supervised Inference

## Data
- Root: `{IMG_DIR}`
- Discovered {len(df)} images

## Model
- Xception (include_top=False) + GAP + Dense(1, sigmoid)
- Preprocess: xception.preprocess_input ([-1,1])
- Weights: `{WEIGHTS_PATH}`

## Labeling (Semi-supervised)
- GMM-based threshold at intersection → auto-label confident tails
- Uncertainty band ±delta to target ~{TARGET_UNCERTAIN} manual labels
- Gradio UI to label only uncertain images
- Final labels: `{OUT_CSV}`

## Supervised Baseline
- Stratified 80/10/10 split on labeled images
- Threshold: chosen by max F1 on validation
- Metrics (validation & test): Accuracy, F1, AUROC, Precision, Recall, Confusion Matrix

## Outputs
- CSVs: frame_level.csv, video_level.csv (id,label,prob_fake,pred_label)
- val_metrics.json, test_metrics.json
- gradcam_samples/ with top 10 confident fakes/reals (validation)
- all_scores.csv, auto_labeled.csv, uncertain_tolabel.csv
- This README.md
"""
with open(f"{OUT_DIR}/README.md","w") as f: f.write(readme)
print("README written to", f"{OUT_DIR}/README.md")

Wrote CSVs: /content/drive/MyDrive/DeepLearning/ffpp_autolabel_out/frame_level.csv | /content/drive/MyDrive/DeepLearning/ffpp_autolabel_out/video_level.csv


/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: [['input_layer_1']]
Received: inputs=['Tensor(shape=(1, 299, 299, 3))']
  warnings.warn(msg)


Saved Grad-CAM samples to: /content/drive/MyDrive/DeepLearning/ffpp_autolabel_out/gradcam_samples
README written to /content/drive/MyDrive/DeepLearning/ffpp_autolabel_out/README.md
